In [2]:
# Since "translation" pipeline is removed in v5.4+, 
# we reuse the same AutoModelForSeq2SeqLM approach:

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── helper: drop-in replacement for the old pipeline("translation") call 
def make_translator(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    def translate(text, max_length=200, **gen_kwargs):
        is_batch = isinstance(text, list)
        texts    = text if is_batch else [text]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=4,
                early_stopping=True,
                **gen_kwargs
            )
        translated = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results = [{"translation_text": t} for t in translated]
        return results if is_batch else results[0]

    return translate


In [4]:

# ── 1. Helsinki-NLP/opus-mt — fast, specific language pairs ──────────────────
#    Lightweight MarianMT models, one model per language pair
#    Best for: production use, low latency, known fixed language pairs
# ─────────────────────────────────────────────────────────────────────────────

# English to French
translate = make_translator("Helsinki-NLP/opus-mt-en-fr")

texts = [
    "Machine learning is transforming the way we interact with technology.",
    "The quarterly earnings report exceeded all analyst expectations.",
    "Please ensure your passport is valid for at least six months before travel.",
]

results = translate(texts)
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"FR : {result['translation_text']}") #type: ignore
    print()

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

EN : Machine learning is transforming the way we interact with technology.
FR : L'apprentissage automatique transforme la façon dont nous interagissons avec la technologie.

EN : The quarterly earnings report exceeded all analyst expectations.
FR : Le rapport trimestriel sur les gains a dépassé toutes les attentes des analystes.

EN : Please ensure your passport is valid for at least six months before travel.
FR : Veuillez vous assurer que votre passeport est valide pendant au moins six mois avant le voyage.



model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

In [5]:
# English to German
translate = make_translator("Helsinki-NLP/opus-mt-en-de")

texts = [
    "The new software update includes several critical security patches.",
    "Renewable energy sources are becoming increasingly cost-competitive.",
    "Our customer support team is available 24 hours a day, seven days a week.",
]

results = translate(texts)
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"DE : {result['translation_text']}") #type:ignore
    print()


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

EN : The new software update includes several critical security patches.
DE : Das neue Software-Update enthält mehrere kritische Sicherheitspatches.

EN : Renewable energy sources are becoming increasingly cost-competitive.
DE : Erneuerbare Energiequellen werden zunehmend kostenwettbewerbsfähiger.

EN : Our customer support team is available 24 hours a day, seven days a week.
DE : Unser Kundensupport-Team steht Ihnen 24 Stunden am Tag, sieben Tage die Woche zur Verfügung.



In [6]:

# English to Hindi
translate = make_translator("Helsinki-NLP/opus-mt-en-hi")

texts = [
    "Artificial intelligence is rapidly changing the healthcare industry.",
    "The government announced a new policy for renewable energy subsidies.",
    "Please submit your application before the deadline on Friday.",
]

results = translate(texts)
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"HI : {result['translation_text']}") #type:ignore
    print()

# EN : Artificial intelligence is rapidly changing the healthcare industry.
# HI : कृत्रिम बुद्धिमत्ता तेजी से स्वास्थ्य सेवा उद्योग को बदल रही है।

# EN : The government announced a new policy for renewable energy subsidies.
# HI : सरकार ने नवीकरणीय ऊर्जा सब्सिडी के लिए एक नई नीति की घोषणा की।

# EN : Please submit your application before the deadline on Friday.
# HI : कृपया शुक्रवार की समय सीमा से पहले अपना आवेदन जमा करें।


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

EN : Artificial intelligence is rapidly changing the healthcare industry.
HI : स्वास्थ्य चिकित्सा उद्योग को तेज़ी से बदल रहा है।

EN : The government announced a new policy for renewable energy subsidies.
HI : सरकार ने नवीकृत ऊर्जा के लिए एक नई नीति घोषित की.

EN : Please submit your application before the deadline on Friday.
HI : शुक्रवार को मृत पंक्ति के पहले कृपया अपने अनुप्रयोग को अधीन करें.



In [7]:
# ── 2. facebook/nllb-200-distilled-600M — 200 languages ──────────────────────
#    Single model covering 200 languages using language codes
#    Best for: less common languages, multilingual pipelines, research
# ─────────────────────────────────────────────────────────────────────────────

# NLLB requires forced_bos_token_id to specify the target language
def make_nllb_translator(model_name="facebook/nllb-200-distilled-600M"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    def translate(text, src_lang, tgt_lang, max_length=200):
        is_batch = isinstance(text, list)
        texts    = text if is_batch else [text]

        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )
        translated = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results = [{"translation_text": t} for t in translated]
        return results if is_batch else results[0]

    return translate

translate = make_nllb_translator("facebook/nllb-200-distilled-600M")

# same source text translated into multiple languages
source = "Climate change is one of the most pressing challenges facing humanity today."

language_pairs = [
    ("eng_Latn", "fra_Latn", "French"),
    ("eng_Latn", "hin_Deva", "Hindi"),
    ("eng_Latn", "ara_Arab", "Arabic"),
    ("eng_Latn", "zho_Hans", "Chinese (Simplified)"),
    ("eng_Latn", "swh_Latn", "Swahili"),
]

print(f"Source (EN): {source}\n")
for src_lang, tgt_lang, lang_name in language_pairs:
    result = translate(source, src_lang=src_lang, tgt_lang=tgt_lang)
    print(f"{lang_name:25s}: {result['translation_text']}") #type:ignore

# Source (EN): Climate change is one of the most pressing challenges facing humanity today.

# French                   : Le changement climatique est l'un des défis les plus pressants auxquels l'humanité est confrontée aujourd'hui.
# Hindi                    : जलवायु परिवर्तन आज मानवता के सामने सबसे गंभीर चुनौतियों में से एक है।
# Arabic                   : يُعدّ تغير المناخ أحد أكثر التحديات التي تواجه البشرية إلحاحاً اليوم.
# Chinese (Simplified)     : 气候变化是当今人类面临的最紧迫挑战之一。
# Swahili                  : Mabadiliko ya tabianchi ni moja ya changamoto kubwa zaidi zinazokabili ubinadamu leo.


# batch: translate multiple sentences at once
texts = [
    "The meeting has been rescheduled to next Monday at 9am.",
    "Please review the attached document and provide your feedback by Thursday.",
    "The project deadline has been extended by two weeks due to scope changes.",
]

results = translate(texts, src_lang="eng_Latn", tgt_lang="fra_Latn")
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"FR : {result['translation_text']}") #type:ignore
    print()

# EN : The meeting has been rescheduled to next Monday at 9am.
# FR : La réunion a été reportée au lundi prochain à 9h.

# EN : Please review the attached document and provide your feedback by Thursday.
# FR : Veuillez examiner le document ci-joint et fournir vos commentaires d'ici jeudi.

# EN : The project deadline has been extended by two weeks due to scope changes.
# FR : La date limite du projet a été prolongée de deux semaines en raison de changements de portée.


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Source (EN): Climate change is one of the most pressing challenges facing humanity today.

French                   : Le changement climatique est l'un des défis les plus pressants auxquels l'humanité est confrontée aujourd'hui.
Hindi                    : जलवायु परिवर्तन आज मानवता के सामने सबसे महत्वपूर्ण चुनौतियों में से एक है।
Arabic                   : Climate change is one of the most pressing challenges facing humanity today.
Chinese (Simplified)     : 气候变化是今天人类面临的最紧迫的挑战之一.
Swahili                  : Mabadiliko ya hali ya hewa ni mojawapo ya changamoto muhimu zaidi zinazowakabili wanadamu leo.
EN : The meeting has been rescheduled to next Monday at 9am.
FR : La réunion a été reportée au lundi prochain à 9 h.

EN : Please review the attached document and provide your feedback by Thursday.
FR : Veuillez examiner le document joint et fournir vos commentaires d'ici jeudi.

EN : The project deadline has been extended by two weeks due to scope changes.
FR : Le délai du projet a été prol

In [8]:
# batch: translate multiple sentences at once
texts = [
    "The meeting has been rescheduled to next Monday at 9am.",
    "Please review the attached document and provide your feedback by Thursday.",
    "The project deadline has been extended by two weeks due to scope changes.",
]

results = translate(texts, src_lang="eng_Latn", tgt_lang="fra_Latn")
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"FR : {result['translation_text']}") #type:ignore
    print()

# EN : The meeting has been rescheduled to next Monday at 9am.
# FR : La réunion a été reportée au lundi prochain à 9h.

# EN : Please review the attached document and provide your feedback by Thursday.
# FR : Veuillez examiner le document ci-joint et fournir vos commentaires d'ici jeudi.

# EN : The project deadline has been extended by two weeks due to scope changes.
# FR : La date limite du projet a été prolongée de deux semaines en raison de changements de portée.

EN : The meeting has been rescheduled to next Monday at 9am.
FR : La réunion a été reportée au lundi prochain à 9 h.

EN : Please review the attached document and provide your feedback by Thursday.
FR : Veuillez examiner le document joint et fournir vos commentaires d'ici jeudi.

EN : The project deadline has been extended by two weeks due to scope changes.
FR : Le délai du projet a été prolongé de deux semaines en raison de changements de portée.



In [9]:


# ── 3. back-translation with Helsinki-NLP ────────────────────────────────────
#    Translate to target language then back to source to verify quality
#    Best for: data augmentation, quality checking, training data generation
# ─────────────────────────────────────────────────────────────────────────────
en_to_es = make_translator("Helsinki-NLP/opus-mt-en-es")
es_to_en = make_translator("Helsinki-NLP/opus-mt-es-en")

originals = [
    "The board of directors approved the merger after months of negotiation.",
    "Early diagnosis significantly improves the chances of successful treatment.",
    "The new regulation requires all companies to disclose their carbon emissions annually.",
]

print("Back-translation quality check (EN -> ES -> EN):\n")
for original in originals:
    spanish    = en_to_es(original)["translation_text"] #type:ignore
    back       = es_to_en(spanish)["translation_text"] #type:ignore
    print(f"Original  : {original}")
    print(f"Spanish   : {spanish}")
    print(f"Back (EN) : {back}")
    print()

# Original  : The board of directors approved the merger after months of negotiation.
# Spanish   : La junta directiva aprobó la fusión después de meses de negociación.
# Back (EN) : The board of directors approved the merger after months of negotiation.

# Original  : Early diagnosis significantly improves the chances of successful treatment.
# Spanish   : El diagnóstico temprano mejora significativamente las posibilidades de un tratamiento exitoso.
# Back (EN) : Early diagnosis significantly improves the chances of successful treatment.

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Back-translation quality check (EN -> ES -> EN):



model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Original  : The board of directors approved the merger after months of negotiation.
Spanish   : El consejo de administración aprobó la fusión después de meses de negociación.
Back (EN) : The Board of Directors approved the merger after months of negotiation.

Original  : Early diagnosis significantly improves the chances of successful treatment.
Spanish   : El diagnóstico temprano mejora significativamente las posibilidades de un tratamiento exitoso.
Back (EN) : Early diagnosis significantly improves the chances of successful treatment.

Original  : The new regulation requires all companies to disclose their carbon emissions annually.
Spanish   : El nuevo reglamento exige que todas las empresas revelen sus emisiones de carbono anualmente.
Back (EN) : The new regulation requires all companies to disclose their carbon emissions annually.



### **text-generation**

In [ ]:
# **Quick reference:**

# | Model | Size | Best for |
# |---|---|---|
# | `gpt2` | 117M | Learning, fast testing, small prompts |
# | `distilgpt2` | 82M | Fastest, edge/resource-constrained |
# | `gpt2-medium/large/xl` | 345M–1.5B | Progressively better GPT-2 quality |
# | `Qwen/Qwen2.5-1.5B` | 1.5B | High quality without GPU |
# | `meta-llama/Llama-3.2-1B` | 1B | Strong general generation, needs HF token |
# | `mistralai/Mistral-7B-v0.1` | 7B | Best quality, requires GPU |

# | Parameter | Effect |
# |---|---|
# | `max_new_tokens` | How many tokens to generate beyond the prompt |
# | `do_sample=False` | Greedy decoding — deterministic, factual |
# | `temperature` | Higher = more creative, lower = more focused |
# | `top_k` | Sample only from top K most likely tokens |
# | `top_p` | Nucleus sampling — cut off at cumulative probability p |
# | `repetition_penalty` | Values above 1.0 reduce word repetition |
# | `return_full_text=False` | Return only generated text, not the prompt |
# | `num_return_sequences` | Number of independent completions to generate |

In [12]:
from transformers import pipeline

# ── 1. gpt2 ──────────────────────────────────────────────────────────────────
#    Classic small model, great for learning and experimentation
#    Best for: understanding generation basics, fast local testing
# ─────────────────────────────────────────────────────────────────────────────

pipe = pipeline("text-generation", model="gpt2")

# basic continuation
result = pipe(
    "The history of the internet began in",
    max_new_tokens=80,       # correct parameter name
    do_sample=False,
    return_full_text=False   # remove temperature when do_sample=False
)

print(result[0]["generated_text"])

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 the early 1990s when the internet was first created. The internet was created by a group of people who wanted to make a better world. The internet was created by people who wanted to make a better world.

The internet was created by a group of people who wanted to make a better world.

The internet was created by a group of people who wanted to make a better world.


In [13]:

# multiple completions with sampling
results = pipe(
    "Scientists have recently discovered that",
    max_new_tokens=60,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    return_full_text=False
)
print("3 different completions:\n")
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r['generated_text'].strip()}")
    print()

# [1] a new species of deep-sea fish off the coast of New Zealand that can
#     survive at depths of over 3,000 metres...
# [2] the human brain forms new neural connections throughout adulthood,
#     challenging the long-held belief that neuroplasticity declines after...
# [3] a chemical compound found in green tea may slow the progression of
#     certain neurodegenerative diseases including Parkinson's...



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


3 different completions:

  [1] the brain activity that makes a person remember a certain story is actually not part of their memory. Research published in the February 18 issue of the Journal of Psychophysiology finds evidence that brain regions that are involved in memory recall are indeed involved in memory formation.

The scientists found that during memory recall

  [2] they don't only produce hydrogen and oxygen but also make it radioactive. The scientists believe this would make it more attractive for them to use the gas to build more weapons.


But the company has also added a new element in its arsenal – iron oxide. It has developed a method for making it

  [3] human genes are responsible for a variety of different kinds of behavior in birds.

The discovery of the new gene sets off several major studies on the genetics of bird behavior. First, researchers discovered that the genetic mechanism behind flight has been altered at different levels in birds. Some of the genes respo

In [14]:
# batch: multiple prompts at once
prompts = [
    "The stock market crashed when",
    "In the year 2045, robots will",
    "The greatest challenge facing modern medicine is",
]

results = pipe(
    prompts,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    return_full_text=False
)

for prompt, result in zip(prompts, results):
    print(f"Prompt : {prompt}")
    print(f"Output : {result[0]['generated_text'].strip()}")
    print()


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt : The stock market crashed when
Output : President Obama won the presidential election. The stock market crashed when Wall Street crashed.

This year's stock market is the one big crash of the summer on the eve of Election Day. A big crash is a big mess. And it is no

Prompt : In the year 2045, robots will
Output : be able to do things such as make medical treatments, monitor patients and more.

But what about that future that will make human and machine collaboration so much more feasible?

"When you have a human robot out there doing things that humans

Prompt : The greatest challenge facing modern medicine is
Output : making sure that we develop more effective treatments. This requires the establishment of scientific, clinical, scientific and technological breakthroughs. To achieve this, we need to find a way to reduce costs.

These are just some of the challenges faced by doctors



In [15]:
# result is a list of dicts, so inspect the first element
print(result)
# [{'generated_text': '...'}]

# keys of the dict
print(result[0].keys())
# dict_keys(['generated_text'])

# each key and value
for k, v in result[0].items():
    print(f"{k}: {v}")
# generated_text: the 1960s, when the United States...

# if you want to inspect the type and structure
print(type(result))        # <class 'list'>
print(type(result[0]))     # <class 'dict'>
print(len(result))         # 1  (or more if num_return_sequences > 1)

[{'generated_text': ' making sure that we develop more effective treatments. This requires the establishment of scientific, clinical, scientific and technological breakthroughs. To achieve this, we need to find a way to reduce costs.\n\nThese are just some of the challenges faced by doctors'}]
dict_keys(['generated_text'])
generated_text:  making sure that we develop more effective treatments. This requires the establishment of scientific, clinical, scientific and technological breakthroughs. To achieve this, we need to find a way to reduce costs.

These are just some of the challenges faced by doctors
<class 'list'>
<class 'dict'>
1


In [16]:
# ── 2. distilgpt2 ────────────────────────────────────────────────────────────
#    Smallest and fastest GPT-2 variant, 40 percent fewer parameters
#    Best for: high-throughput, edge deployment, resource-constrained systems
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("text-generation", model="distilgpt2")

# creative writing
prompts = [
    "Deep in the forest, an old man discovered",
    "The last astronaut on the space station looked out the window and saw",
    "She opened the letter and read the words that would change everything:",
]

for prompt in prompts:
    result = pipe(
        prompt,
        max_new_tokens=70,
        do_sample=True,
        temperature=0.85,
        top_p=0.92,
        repetition_penalty=1.3,
        return_full_text=False
    )
    print(f"Prompt : {prompt}")
    print(f"Output : {result[0]['generated_text'].strip()}")
    print()

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt : Deep in the forest, an old man discovered
Output : something very strange inside him. He had been hiding there for long without any notice to anyone and he was not sure what happened that night but it seemed like a lot of his life would be lost if you didn't keep silent about this mysterious figure."
"I don´t want to lose anything," said Olukana while looking at her



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt : The last astronaut on the space station looked out the window and saw
Output : a man standing next to him.
“This is what you see, my favorite photo of any ISS crew member in history? I thought that would be great." ‟"You can take pictures from all this while sitting there or watching movies — so it's just an excuse for getting up here … maybe one more time before going down somewhere

Prompt : She opened the letter and read the words that would change everything:
Output : “The world is a lot more complicated than it used to be.”
But for those unfamiliar with China, all these things don't mean nothing in any way—it's not something we've seen from America since President Obama took office last year or even just one day before this election cycle — but if you want to know what I



In [17]:
# ── 3. Qwen/Qwen2.5-1.5B ─────────────────────────────────────────────────────
#    Efficient 1.5B model from Alibaba, much stronger than GPT-2 at same size
#    Best for: higher quality output without requiring a GPU
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("text-generation",
                model="Qwen/Qwen2.5-1.5B",
                device_map="auto")

# factual continuation
prompts = [
    "The key differences between supervised and unsupervised learning are",
    "To build a REST API in Python using FastAPI, you need to",
    "The main causes of the 2008 financial crisis were",
]

for prompt in prompts:
    result = pipe(
        prompt,
        max_new_tokens=120,
        do_sample=False,           # greedy for factual content
        repetition_penalty=1.2,
        return_full_text=False
    )
    print(f"Prompt : {prompt}")
    print(f"Output : {result[0]['generated_text'].strip()}")
    print()

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Prompt : The key differences between supervised and unsupervised learning are
Output : as follows:
A. Supervised learning requires the use of labeled data, while unsupervised learning does not.
B. In supervised learning tasks, there is a need to predict continuous values; in contrast, for unsupervised learning problems, it's necessary to identify discrete categories or clusters from unlabeled datasets.
C. The output results produced by supervised learning can be directly used without further processing (e.g., classification), whereas this isn't true for unsupervised learning outputs which often require additional steps such as feature extraction before they can be applied effectively.

Please select all correct descriptions above that accurately

Prompt : To build a REST API in Python using FastAPI, you need to
Output : follow these steps:

1. Install the required packages: `fastapi`, `uvicorn` and `pydantic`.
2. Create an application class that inherits from `BaseApplication`. This wi

In [18]:
# ── 4. generation parameters comparison ──────────────────────────────────────
#    Shows how temperature and sampling strategy affect output style
#    Same prompt, four different parameter configurations
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("text-generation", model="gpt2")

prompt = "The future of renewable energy looks"

configs = [
    {
        "label":       "Greedy (deterministic)",
        "do_sample":   False,
        "temperature": 1.0,
    },
    {
        "label":       "Low temperature (focused)",
        "do_sample":   True,
        "temperature": 0.3,
        "top_k":       40,
    },
    {
        "label":       "Medium temperature (balanced)",
        "do_sample":   True,
        "temperature": 0.8,
        "top_p":       0.92,
    },
    {
        "label":       "High temperature (creative)",
        "do_sample":   True,
        "temperature": 1.4,
        "top_k":       100,
        "top_p":       0.98,
    },
]

print(f"Prompt: {prompt}\n")
for config in configs:
    label = config.pop("label")
    result = pipe(
        prompt,
        max_new_tokens=60,
        return_full_text=False,
        repetition_penalty=1.2,
        **config
    )
    print(f"  [{label}]")
    print(f"  {result[0]['generated_text'].strip()}")
    print()

# [Greedy (deterministic)]
#   bright, with solar and wind power expected to account for more than
#   50 percent of global electricity generation by 2035...

# [Low temperature (focused)]
#   very promising, with solar capacity growing faster than any other energy
#   source in history. Battery storage costs have fallen dramatically...

# [Medium temperature (balanced)]
#   increasingly competitive, driven by falling costs and strong government
#   policy support. Offshore wind projects are being developed at a scale
#   that was unimaginable just a decade ago...

# [High temperature (creative)]
#   stranger than anyone predicted — not the clean utopia of magazine covers
#   but something messier, political, full of unexpected winners and losers
#   fighting over land, water, minerals, and grid access...

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: The future of renewable energy looks

  [Greedy (deterministic)]
  bright for the United States, but it's not going to be easy.

  [Low temperature (focused)]
  bright.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  [Medium temperature (balanced)]
  bleak.

  [High temperature (creative)]
  bright in its own right, according to one economist
 - David Rosebud (author), The National Interest : Renewableenergy development has evolved profoundly thanks with enormous financial aid... we are on the road towards our natural means – human capital — for some 45 years out...



In [19]:
# Since "text2text-generation" is also removed in v5.4+, 
# we reuse the same AutoModelForSeq2SeqLM approach:

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── helper: drop-in replacement for pipeline("text2text-generation") ──────────
def make_text2text(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    def generate(text, max_new_tokens=100, min_new_tokens=5, **gen_kwargs):
        is_batch = isinstance(text, list)
        texts    = text if is_batch else [text]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                num_beams=4,
                early_stopping=True,
                **gen_kwargs
            )
        outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results = [{"generated_text": o} for o in outputs]
        return results if is_batch else results[0]

    return generate


# ── 1. google/flan-t5-base ────────────────────────────────────────────────────
#    Instruction-following T5, handles many tasks via natural language prompts
#    Best for: multi-task use, QA, summarization, translation, grammar all in one
# ─────────────────────────────────────────────────────────────────────────────
generate = make_text2text("google/flan-t5-base")

# QA style
qa_prompts = [
    "question: Who invented the telephone? context: Alexander Graham Bell is credited "
    "with inventing the telephone in 1876. He was a Scottish-born scientist and inventor.",

    "question: What is the boiling point of water? context: Water boils at 100 degrees "
    "Celsius or 212 degrees Fahrenheit at standard atmospheric pressure.",

    "question: Which company makes the iPhone? context: Apple Inc. is an American "
    "multinational technology company that designs and sells the iPhone smartphone.",
]

print("── QA style ──")
for prompt in qa_prompts:
    result = generate(prompt, max_new_tokens=30)
    print(f"Q      : {prompt.split('?')[0].replace('question: ', '')}")
    print(f"Answer : {result['generated_text']}") # type: ignore
    print()

# Q      : Who invented the telephone
# Answer : Alexander Graham Bell

# Q      : What is the boiling point of water
# Answer : 100 degrees Celsius

# Q      : Which company makes the iPhone
# Answer : Apple Inc.


# summarization style
summarize_prompts = [
    "summarize: The European Union was founded in 1993 following the signing of the "
    "Maastricht Treaty. It currently comprises 27 member states and operates a single "
    "market allowing the free movement of goods, services, capital, and people. The EU "
    "has its own parliament, a central bank, and a common currency used by 20 of its members.",

    "summarize: Photosynthesis is the process by which plants use sunlight, water, and "
    "carbon dioxide to produce oxygen and energy in the form of glucose. It takes place "
    "mainly in the leaves, where chlorophyll absorbs light energy. This process is "
    "fundamental to life on Earth as it forms the base of almost all food chains.",
]

print("── Summarization style ──")
for prompt in summarize_prompts:
    result = generate(prompt, max_new_tokens=60, min_new_tokens=10)
    print(f"Summary : {result['generated_text']}") # type: ignore
    print()

# Summary : The European Union was founded in 1993 and has 27 member states.
#           It operates a single market and has its own parliament and currency.

# Summary : Photosynthesis is the process by which plants use sunlight and water
#           to produce oxygen and glucose, forming the base of most food chains.


# translation style
translation_prompts = [
    "translate English to French: The meeting has been rescheduled to Thursday morning.",
    "translate English to German: Please submit your report before the end of the week.",
    "translate English to Spanish: The new product launch exceeded all sales expectations.",
]

print("── Translation style ──")
for prompt in translation_prompts:
    result = generate(prompt, max_new_tokens=60)
    print(f"EN : {prompt.replace('translate English to French: ', '').replace('translate English to German: ', '').replace('translate English to Spanish: ', '')}")
    print(f"OUT: {result['generated_text']}") # type: ignore
    print()

# EN : The meeting has been rescheduled to Thursday morning.
# OUT: La réunion a été replanifiée à jeudi matin.

# EN : Please submit your report before the end of the week.
# OUT: Bitte reichen Sie Ihren Bericht vor Ende der Woche ein.

# EN : The new product launch exceeded all sales expectations.
# OUT: El lanzamiento del nuevo producto superó todas las expectativas de ventas.


# grammar correction style
grammar_prompts = [
    "Fix grammar: She go to the market every day and buys vegetable.",
    "Fix grammar: He don't knows the answer to the question what teacher asked.",
    "Fix grammar: They was very excited about the news what they heard yesterday.",
]

print("── Grammar correction style ──")
for prompt in grammar_prompts:
    result = generate(prompt, max_new_tokens=60)
    original = prompt.replace("Fix grammar: ", "")
    print(f"Original : {original}")
    print(f"Fixed    : {result['generated_text']}") # type: ignore
    print()

# Original : She go to the market every day and buys vegetable.
# Fixed    : She goes to the market every day and buys vegetables.

# Original : He don't knows the answer to the question what teacher asked.
# Fixed    : He doesn't know the answer to the question the teacher asked.

# Original : They was very excited about the news what they heard yesterday.
# Fixed    : They were very excited about the news they heard yesterday.


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

── QA style ──
Q      : Who invented the telephone
Answer : Alexander Graham Bell is credited with inventing the telephone

Q      : What is the boiling point of water
Answer : 212 degrees Fahrenheit

Q      : Which company makes the iPhone
Answer : Apple Inc. is an American multinational technology company

── Summarization style ──
Summary : The European Union is a member state of the European Union.

Summary : Plants use photosynthesis to produce oxygen and energy in the form of glucose.

── Translation style ──
EN : The meeting has been rescheduled to Thursday morning.
OUT: La séance a été reschedulée à l'heure de l'ouverture du jeudi.

EN : Please submit your report before the end of the week.
OUT: Bitte schicken Sie Ihre Bericht vor dem Ende des Wochen.

EN : The new product launch exceeded all sales expectations.
OUT: El nuevo lanzamiento de productos supera todos los expectativos de venta.

── Grammar correction style ──
Original : She go to the market every day and buys vegeta

In [21]:
# ── 2. t5-base ────────────────────────────────────────────────────────────────
#    Original T5, uses specific prefix format for each task
#    Best for: translation and summarization with the original T5 prefix style
# ─────────────────────────────────────────────────────────────────────────────
generate = make_text2text("t5-base")

# t5-base uses specific prefixes: "summarize:", "translate English to German:", etc.
prompts = [
    (
        "summarize:",
        "NASA's James Webb Space Telescope has captured the deepest and sharpest "
        "infrared image of the distant universe to date. The image shows thousands "
        "of galaxies, including the faintest objects ever observed, some of which "
        "formed less than a billion years after the Big Bang. The telescope uses "
        "a 6.5 metre primary mirror and instruments cooled to near absolute zero "
        "to detect infrared light from these extremely distant and ancient galaxies."
    ),
    (
        "translate English to German:",
        "Artificial intelligence is rapidly changing the way businesses operate "
        "and how people interact with technology in their daily lives."
    ),
    (
        "translate English to French:",
        "The quarterly financial results showed a 12 percent increase in revenue "
        "compared to the same period last year."
    ),
]

print("── t5-base with task prefixes ──")
for prefix, text in prompts:
    prompt = f"{prefix} {text}"
    result = generate(prompt, max_new_tokens=80)
    print(f"Task   : {prefix}")
    print(f"Input  : {text[:80]}...")
    print(f"Output : {result['generated_text']}") # type: ignore
    print()

# Task   : summarize:
# Input  : NASA's James Webb Space Telescope has captured the deepest and sharpest...
# Output : James Webb Space Telescope has captured the deepest infrared image of
#          the distant universe, showing thousands of galaxies formed after the Big Bang.

# Task   : translate English to German:
# Input  : Artificial intelligence is rapidly changing the way businesses operate...
# Output : Künstliche Intelligenz verändert schnell die Geschäftstätigkeit und die
#          Art und Weise, wie Menschen täglich mit Technologie interagieren.

# Task   : translate English to French:
# Input  : The quarterly financial results showed a 12 percent increase in revenue...
# Output : Les résultats financiers trimestriels ont montré une augmentation

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

── t5-base with task prefixes ──
Task   : summarize:
Input  : NASA's James Webb Space Telescope has captured the deepest and sharpest infrared...
Output : the image shows thousands of galaxies, including the faintest objects ever observed, some of which formed less than a billion years after the Big Bang .

Task   : translate English to German:
Input  : Artificial intelligence is rapidly changing the way businesses operate and how p...
Output : Künstliche Intelligenz verändert die Art und Weise, wie Unternehmen arbeiten und wie Menschen mit Technologie im täglichen Leben interagieren.

Task   : translate English to French:
Input  : The quarterly financial results showed a 12 percent increase in revenue compared...
Output : Les résultats financiers trimestriels ont fait apparaître une augmentation de 12 p. 100 des recettes par rapport à la même période l’an dernier.



In [22]:
# ── 3. batch processing — multiple tasks in one call ─────────────────────────
#    Send different task types in a single batch
#    Best for: mixed workloads, efficient GPU utilisation
# ─────────────────────────────────────────────────────────────────────────────
generate = make_text2text("google/flan-t5-base")

mixed_batch = [
    "question: What year did World War II end? context: World War II ended in 1945 "
    "with the surrender of Germany in May and Japan in September.",

    "summarize: The Python programming language was created by Guido van Rossum and "
    "first released in 1991. It emphasises code readability and simplicity, and supports "
    "multiple programming paradigms including procedural, object-oriented, and functional.",

    "translate English to Spanish: Please confirm your attendance by replying to this email.",

    "Fix grammar: The childrens was playing in the park when it start to rain heavily.",

    "question: What is the speed of light? context: Light travels at approximately "
    "299,792 kilometres per second in a vacuum, often approximated as 300,000 km/s.",
]

print("── Mixed batch (different tasks in one call) ──")
results = generate(mixed_batch, max_new_tokens=60)
task_labels = ["QA", "Summarize", "Translate", "Grammar", "QA"]

for label, prompt, result in zip(task_labels, mixed_batch, results):
    print(f"[{label}]")
    print(f"  Input  : {prompt[:70]}...")
    print(f"  Output : {result['generated_text']}") # type: ignore
    print()

# [QA]
#   Input  : question: What year did World War II end?...
#   Output : 1945

# [Summarize]
#   Input  : summarize: The Python programming language was created by Guido van Rossum...
#   Output : Python was created by Guido van Rossum in 1991 and supports multiple
#            programming paradigms including procedural and object-oriented.

# [Translate]
#   Input  : translate English to Spanish: Please confirm your attendance...
#   Output : Por favor, confirme su asistencia respondiendo a este correo electrónico.

# [Grammar]
#   Input  : Fix grammar: The childrens was playing in the park...
#   Output : The children were playing in the park when it started to rain heavily.

# [QA]
#   Input  : question: What is the speed of light?...
#   Output : 299,792 kilometres per second

── Mixed batch (different tasks in one call) ──
[QA]
  Input  : question: What year did World War II end? context: World War II ended ...
  Output : 1945 with the surrender of Germany in May and Japan

[Summarize]
  Input  : summarize: The Python programming language was created by Guido van Ro...
  Output : Python is a programming language developed by Guido van Rossum.

[Translate]
  Input  : translate English to Spanish: Please confirm your attendance by replyi...
  Output : Por favor confirmar tu acuerdo en respuesta a esta envo.

[Grammar]
  Input  : Fix grammar: The childrens was playing in the park when it start to ra...
  Output : The childrens was playing in the park when it start to rain heavily.

[QA]
  Input  : question: What is the speed of light? context: Light travels at approx...
  Output : 300,000 km/s



#### **fill-mask**

Predict what word should fill a masked position in a sentence. Used to understand what a model has learned about language, and for probing model knowledge.



In [24]:

# | Model | Mask token | Best for |
# |---|---|---|
# | `bert-base-uncased` | `[MASK]` | General English, factual probing |
# | `bert-base-cased` | `[MASK]` | Proper nouns, named entities |
# | `roberta-base` | `<mask>` | Higher accuracy than BERT |
# | `xlm-roberta-base` | `<mask>` | 100 languages, multilingual |
# | `distilbert-base-uncased` | `[MASK]` | Fast, high-throughput |

# | Parameter | Effect |
# |---|---|
# | `top_k=5` | Return top 5 predictions instead of default 5 |
# | `targets=[...]` | Restrict predictions to only these specific tokens |
# | `score` | Confidence of that token filling the mask |
# | `sequence` | Full sentence with the mask replaced |
# | `token_str` | The predicted word itself |

In [23]:
from transformers import pipeline

# ── 1. bert-base-uncased ──────────────────────────────────────────────────────
#    Standard English BERT, uncased (lowercase everything)
#    Best for: general English, factual probing, understanding model knowledge
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("fill-mask", model="bert-base-uncased")

# factual knowledge probing
prompts = [
    "The capital of Japan is [MASK].",
    "Water freezes at [MASK] degrees Celsius.",
    "The [MASK] is the largest organ in the human body.",
    "Albert Einstein developed the theory of [MASK].",
    "The Great Wall of [MASK] is visible from space.",
]

print("── Factual probing ──")
for prompt in prompts:
    results = pipe(prompt, top_k=3)
    print(f"Prompt : {prompt}")
    for r in results:
        print(f"  [{r['score']:.4f}]  {r['token_str']:15s}  →  {r['sequence']}")
    print()

# Prompt : The capital of Japan is [MASK].
#   [0.9823]  tokyo            →  the capital of japan is tokyo.
#   [0.0041]  osaka            →  the capital of japan is osaka.
#   [0.0018]  kyoto            →  the capital of japan is kyoto.

# Prompt : Water freezes at [MASK] degrees Celsius.
#   [0.8921]  0                →  water freezes at 0 degrees celsius.
#   [0.0412]  zero             →  water freezes at zero degrees celsius.
#   [0.0231]  -10              →  water freezes at -10 degrees celsius.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


── Factual probing ──
Prompt : The capital of Japan is [MASK].
  [0.5642]  tokyo            →  the capital of japan is tokyo.
  [0.0945]  osaka            →  the capital of japan is osaka.
  [0.0914]  kyoto            →  the capital of japan is kyoto.

Prompt : Water freezes at [MASK] degrees Celsius.
  [0.0435]  100              →  water freezes at 100 degrees celsius.
  [0.0418]  60               →  water freezes at 60 degrees celsius.
  [0.0405]  50               →  water freezes at 50 degrees celsius.

Prompt : The [MASK] is the largest organ in the human body.
  [0.4318]  heart            →  the heart is the largest organ in the human body.
  [0.1665]  organ            →  the organ is the largest organ in the human body.
  [0.1165]  brain            →  the brain is the largest organ in the human body.

Prompt : Albert Einstein developed the theory of [MASK].
  [0.9418]  relativity       →  albert einstein developed the theory of relativity.
  [0.0091]  gravity          →  albert e

In [25]:

# targets — restrict predictions to a specific set of tokens
print("── targets parameter ──")
sentiment_prompts = [
    ("The movie was absolutely [MASK].",     ["brilliant", "terrible", "average"]),
    ("I feel [MASK] about the news today.",  ["happy", "sad", "anxious", "excited"]),
    ("The food at this restaurant is [MASK].",["good", "bad", "overpriced", "delicious"])
]

for prompt, targets in sentiment_prompts:
    # strip any accidental leading quote in targets
    clean_targets = [t.lstrip('"') for t in targets]
    results = pipe(prompt, targets=clean_targets)
    print(f"Prompt  : {prompt}")
    print(f"Targets : {clean_targets}")
    for r in results:
        print(f"  [{r['score']:.4f}]  {r['token_str']}")
    print()

# Prompt  : The movie was absolutely [MASK].
# Targets : ['brilliant', 'terrible', 'average']
#   [0.7823]  brilliant
#   [0.1456]  terrible
#   [0.0721]  average

# Prompt  : I feel [MASK] about the news today.
# Targets : ['happy', 'sad', 'anxious', 'excited']
#   [0.4123]  anxious
#   [0.3012]  sad
#   [0.1834]  happy
#   [0.1031]  excited


The specified target token `overpriced` does not exist in the model vocabulary. Replacing with `over`.


── targets parameter ──
Prompt  : The movie was absolutely [MASK].
Targets : ['brilliant', 'terrible', 'average']
  [0.0182]  brilliant
  [0.0085]  terrible
  [0.0003]  average

Prompt  : I feel [MASK] about the news today.
Targets : ['happy', 'sad', 'anxious', 'excited']
  [0.0077]  excited
  [0.0045]  happy
  [0.0031]  anxious
  [0.0029]  sad

Prompt  : The food at this restaurant is [MASK].
Targets : ['good', 'bad', 'overpriced', 'delicious']
  [0.0951]  delicious
  [0.0713]  good
  [0.0008]  bad
  [0.0000]  over



In [26]:
# ── 2. bert-base-cased ────────────────────────────────────────────────────────
#    Case-sensitive BERT — preserves proper nouns, names, acronyms
#    Best for: named entities, proper nouns, code, case-sensitive text
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("fill-mask", model="bert-base-cased")

prompts = [
    "Elon [MASK] is the CEO of Tesla and SpaceX.",
    "[MASK] was the first person to walk on the moon.",
    "The headquarters of [MASK] is located in Cupertino, California.",
    "The [MASK] Prize in Physics was awarded to Albert Einstein in 1921.",
]

print("── Case-sensitive probing (proper nouns) ──")
for prompt in prompts:
    results = pipe(prompt, top_k=3)
    print(f"Prompt : {prompt}")
    for r in results:
        print(f"  [{r['score']:.4f}]  {r['token_str']:15s}  →  {r['sequence']}")
    print()

# Prompt : Elon [MASK] is the CEO of Tesla and SpaceX.
#   [0.9734]  Musk             →  Elon Musk is the CEO of Tesla and SpaceX.
#   [0.0089]  Morley           →  Elon Morley is the CEO of Tesla and SpaceX.
#   [0.0041]  Johnson          →  Elon Johnson is the CEO of Tesla and SpaceX.

# Prompt : [MASK] was the first person to walk on the moon.
#   [0.8921]  Armstrong        →  Armstrong was the first person to walk on the moon.
#   [0.0412]  Aldrin           →  Aldrin was the first person to walk on the moon.
#   [0.0231]  NASA             →  NASA was the first person to walk on the moon.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


── Case-sensitive probing (proper nouns) ──
Prompt : Elon [MASK] is the CEO of Tesla and SpaceX.
  [0.0234]  Jones            →  Elon Jones is the CEO of Tesla and SpaceX.
  [0.0233]  Smith            →  Elon Smith is the CEO of Tesla and SpaceX.
  [0.0174]  Johnson          →  Elon Johnson is the CEO of Tesla and SpaceX.

Prompt : [MASK] was the first person to walk on the moon.
  [0.3053]  He               →  He was the first person to walk on the moon.
  [0.2442]  I                →  I was the first person to walk on the moon.
  [0.2290]  She              →  She was the first person to walk on the moon.

Prompt : The headquarters of [MASK] is located in Cupertino, California.
  [0.0401]  IBM              →  The headquarters of IBM is located in Cupertino, California.
  [0.0325]  Microsoft        →  The headquarters of Microsoft is located in Cupertino, California.
  [0.0236]  BP               →  The headquarters of BP is located in Cupertino, California.

Prompt : The [MASK] Prize i

In [27]:
# ── 3. roberta-base ───────────────────────────────────────────────────────────
#    Stronger than BERT, uses <mask> token instead of [MASK]
#    Best for: higher accuracy predictions, nuanced language understanding
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("fill-mask", model="roberta-base")

# note: RoBERTa uses <mask> not [MASK]
prompts = [
    "The patient was diagnosed with <mask> and prescribed antibiotics.",
    "The defendant was found <mask> by the jury after a three-week trial.",
    "She submitted her <mask> to the university three days before the deadline.",
    "The company announced a <mask> of 5,000 employees due to restructuring.",
]

print("── RoBERTa predictions (<mask> token) ──")
for prompt in prompts:
    results = pipe(prompt, top_k=4)
    print(f"Prompt : {prompt}")
    for r in results:
        print(f"  [{r['score']:.4f}]  {r['token_str']:15s}")
    print()

# Prompt : The patient was diagnosed with <mask> and prescribed antibiotics.
#   [0.3421]  pneumonia
#   [0.1823]  an infection
#   [0.1234]  bronchitis
#   [0.0912]  strep throat

# Prompt : The defendant was found <mask> by the jury after a three-week trial.
#   [0.7823]  guilty
#   [0.2041]  not guilty
#   [0.0089]  innocent
#   [0.0047]  liable


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


── RoBERTa predictions (<mask> token) ──
Prompt : The patient was diagnosed with <mask> and prescribed antibiotics.
  [0.5247]   pneumonia     
  [0.1123]   cancer        
  [0.0652]   leukemia      
  [0.0355]   diabetes      

Prompt : The defendant was found <mask> by the jury after a three-week trial.
  [0.9714]   guilty        
  [0.0232]   innocent      
  [0.0025]   competent     
  [0.0009]   incompetent   

Prompt : She submitted her <mask> to the university three days before the deadline.
  [0.5900]   application   
  [0.0716]   resignation   
  [0.0269]   complaint     
  [0.0190]   proposal      

Prompt : The company announced a <mask> of 5,000 employees due to restructuring.
  [0.8203]   reduction     
  [0.0928]   loss          
  [0.0451]   cut           
  [0.0052]   decrease      



In [28]:
# ── 4. xlm-roberta-base ───────────────────────────────────────────────────────
#    Multilingual model covering 100 languages, uses <mask> token
#    Best for: non-English text, cross-lingual probing
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("fill-mask", model="xlm-roberta-base")

multilingual_prompts = [
    ("English", "The president signed the <mask> into law yesterday."),
    ("French",  "La capitale de la France est <mask>."),
    ("German",  "Berlin ist die Hauptstadt von <mask>."),
    ("Spanish", "El idioma oficial de México es el <mask>."),
    ("Hindi",   "भारत की राजधानी <mask> है।"),
]

print("── Multilingual fill-mask ──")
for lang, prompt in multilingual_prompts:
    results = pipe(prompt, top_k=3)
    print(f"[{lang}] {prompt}")
    for r in results:
        print(f"  [{r['score']:.4f}]  {r['token_str']}")
    print()

# [English] The president signed the <mask> into law yesterday.
#   [0.3421]  bill
#   [0.2134]  legislation
#   [0.1823]  agreement

# [French] La capitale de la France est <mask>.
#   [0.9412]  Paris
#   [0.0234]  Lyon
#   [0.0089]  Marseille

# [German] Berlin ist die Hauptstadt von <mask>.
#   [0.9234]  Deutschland
#   [0.0412]  Preußen
#   [0.0123]  Europa

# [Spanish] El idioma oficial de México es el <mask>.
#   [0.8923]  español
#   [0.0634]  inglés
#   [0.0234]  castellano

# [Hindi] भारत की राजधानी <mask> है।
#   [0.9123]  दिल्ली
#   [0.0412]  मुंबई
#   [0.0234]  कोलकाता


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


── Multilingual fill-mask ──
[English] The president signed the <mask> into law yesterday.
  [0.5160]  bill
  [0.0644]  deal
  [0.0530]  measure

[French] La capitale de la France est <mask>.
  [0.4154]  Paris
  [0.0566]  Bruxelles
  [0.0505]  capitale

[German] Berlin ist die Hauptstadt von <mask>.
  [0.5058]  Deutschland
  [0.1656]  Europa
  [0.1443]  Brandenburg

[Spanish] El idioma oficial de México es el <mask>.
  [0.3574]  inglés
  [0.2456]  español
  [0.1214]  Español

[Hindi] भारत की राजधानी <mask> है।
  [0.5786]  दिल्ली
  [0.0572]  कोलकाता
  [0.0488]  लखनऊ



In [29]:
# ── 5. distilbert-base-uncased ────────────────────────────────────────────────
#    Smaller, 40 percent faster BERT with similar performance
#    Best for: high-throughput, latency-sensitive systems
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("fill-mask", model="distilbert-base-uncased")

# batch: multiple sentences at once
prompts = [
    "Python is a popular programming [MASK] used in data science.",
    "The [MASK] of gravity on Earth is approximately 9.8 m/s squared.",
    "A [MASK] is a collection of key-value pairs in Python.",
    "The human body has [MASK] bones in total.",
]

print("── Batch fill-mask ──")
results = pipe(prompts, top_k=3)

for prompt, preds in zip(prompts, results):
    print(f"Prompt : {prompt}")
    for r in preds:
        print(f"  [{r['score']:.4f}]  {r['token_str']:15s}")
    print()

# Prompt : Python is a popular programming [MASK] used in data science.
#   [0.6823]  language
#   [0.1234]  tool
#   [0.0912]  framework

# Prompt : The [MASK] of gravity on Earth is approximately 9.8 m/s squared.
#   [0.7234]  force
#   [0.1823]  acceleration
#   [0.0634]  rate

# Prompt : A [MASK] is a collection of key-value pairs in Python.
#   [0.8234]  dictionary
#   [0.0912]  dict
#   [0.0423]  tuple

# Prompt : The human body has [MASK] bones in total.
#   [0.5423]  206
#   [0.1234]  many
#   [0.0823]  numerous

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


── Batch fill-mask ──
Prompt : Python is a popular programming [MASK] used in data science.
  [0.9787]  language       
  [0.0061]  tool           
  [0.0059]  technique      

Prompt : The [MASK] of gravity on Earth is approximately 9.8 m/s squared.
  [0.2816]  acceleration   
  [0.2025]  density        
  [0.0670]  velocity       

Prompt : A [MASK] is a collection of key-value pairs in Python.
  [0.0461]  list           
  [0.0425]  database       
  [0.0272]  set            

Prompt : The human body has [MASK] bones in total.
  [0.0060]  181            
  [0.0056]  216            
  [0.0051]  260            



"zero-shot-classification" works out of the box — no fine-tuning needed:

In [1]:
from transformers import pipeline

# ── 1. facebook/bart-large-mnli ──────────────────────────────────────────────
#    Best general-purpose zero-shot classifier
#    Best for: broad topic classification, intent detection, content routing
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# basic single-label classification
prompts = [
    ("I want to book a flight to Paris.",                    ["travel", "cooking", "sports", "technology"]),
    ("The stock market dropped by 3 percent today.",         ["finance", "sports", "health", "entertainment"]),
    ("She submitted her thesis on quantum entanglement.",    ["science", "politics", "food", "fashion"]),
    ("Rohit scored a century in the last over.",             ["cricket", "technology", "cooking", "travel"]),
    ("The new iPhone has a 48MP camera and USB-C port.",     ["technology", "health", "sports", "politics"]),
]

print("── Single-label classification ──")
for text, labels in prompts:
    result = pipe(text, candidate_labels=labels)
    print(f"Text   : {text}")
    for label, score in zip(result["labels"], result["scores"]): # type: ignore
        print(f"  [{score:.4f}]  {label}")
    print()

Device set to use cpu


── Single-label classification ──
Text   : I want to book a flight to Paris.
  [0.9677]  travel
  [0.0236]  technology
  [0.0055]  sports
  [0.0031]  cooking

Text   : The stock market dropped by 3 percent today.
  [0.9349]  finance
  [0.0361]  entertainment
  [0.0163]  health
  [0.0128]  sports

Text   : She submitted her thesis on quantum entanglement.
  [0.9801]  science
  [0.0078]  fashion
  [0.0065]  politics
  [0.0057]  food

Text   : Rohit scored a century in the last over.
  [0.5924]  cricket
  [0.1803]  technology
  [0.1688]  travel
  [0.0585]  cooking

Text   : The new iPhone has a 48MP camera and USB-C port.
  [0.9630]  technology
  [0.0170]  health
  [0.0154]  sports
  [0.0046]  politics



In [2]:
# ── 2. multi_label=True ───────────────────────────────────────────────────────
#    Each label is scored independently — text can belong to multiple categories
#    Best for: tagging, content moderation, multi-topic documents
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

multilabel_prompts = [
    (
        "I love cooking Italian food while watching the IPL match.",
        ["food", "sports", "travel", "music", "technology"]
    ),
    (
        "Drishya is learning PyTorch and also training for a half-marathon.",
        ["machine learning", "fitness", "cooking", "finance"]
    ),
    (
        "The government announced new AI regulations and tax relief for startups.",
        ["technology", "politics", "finance", "health", "sports"]
    ),
]

print("── Multi-label classification (multi_label=True) ──")
for text, labels in multilabel_prompts:
    result = pipe(text, candidate_labels=labels, multi_label=True)
    print(f"Text   : {text}")
    for label, score in zip(result["labels"], result["scores"]): # type: ignore
        print(f"  [{score:.4f}]  {label}")
    print()

# Text   : I love cooking Italian food while watching the IPL match.
#   [0.9234]  food
#   [0.8912]  sports
#   [0.1023]  travel
#   [0.0412]  music
#   [0.0234]  technology

# Text   : Drishya is learning PyTorch and also training for a half-marathon.
#   [0.9512]  machine learning
#   [0.9134]  fitness
#   [0.0423]  finance
#   [0.0189]  cooking

# Text   : The government announced new AI regulations and tax relief for startups.
#   [0.9623]  technology
#   [0.9234]  politics
#   [0.8912]  finance
#   [0.0412]  health
#   [0.0123]  sports

Device set to use cpu


── Multi-label classification (multi_label=True) ──
Text   : I love cooking Italian food while watching the IPL match.
  [0.9931]  sports
  [0.9618]  food
  [0.0018]  technology
  [0.0005]  travel
  [0.0003]  music

Text   : Drishya is learning PyTorch and also training for a half-marathon.
  [0.9892]  fitness
  [0.8632]  machine learning
  [0.0008]  finance
  [0.0005]  cooking

Text   : The government announced new AI regulations and tax relief for startups.
  [0.9744]  technology
  [0.1382]  finance
  [0.0043]  politics
  [0.0009]  health
  [0.0002]  sports



In [4]:
# ── 3. hypothesis_template ────────────────────────────────────────────────────
#    Controls how the model frames each candidate label internally
#    Default: "This example is {}." — override for domain-specific phrasing
#    Best for: sentiment, intent, formality, domain-specific classification
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

hypothesis_examples = [
    (
        "The product broke after two days. Worst purchase ever.",
        ["positive review", "negative review", "neutral review"],
        "This is a {}."
    ),
    (
        "Could you please reschedule our meeting to Thursday?",
        ["request", "complaint", "greeting", "farewell"],
        "The intent of this message is a {}."
    ),
    (
        "Pursuant to the aforementioned clause, the party of the first part shall indemnify.",
        ["legal document", "casual conversation", "social media post", "academic paper"],
        "This text belongs to the category of {}."
    ),
    (
        "Bhai kal match tha, kya scene tha yaar.",
        ["formal communication", "informal conversation", "technical documentation"],
        "This is an example of {}."
    ),
]

print("── hypothesis_template ──")
for text, labels, template in hypothesis_examples:
    result = pipe(text, candidate_labels=labels, hypothesis_template=template)
    print(f"Text     : {text}")
    print(f"Template : {template}")
    for label, score in zip(result["labels"], result["scores"]): #type: ignore
        print(f"  [{score:.4f}]  {label}")
    print()

# Text     : The product broke after two days. Worst purchase ever.
# Template : This is a {}.
#   [0.9712]  negative review
#   [0.0198]  neutral review
#   [0.0090]  positive review

# Text     : Could you please reschedule our meeting to Thursday?
# Template : The intent of this message is a {}.
#   [0.9134]  request
#   [0.0512]  greeting
#   [0.0234]  complaint
#   [0.0120]  farewell


Device set to use cpu


── hypothesis_template ──
Text     : The product broke after two days. Worst purchase ever.
Template : This is a {}.
  [0.9899]  negative review
  [0.0071]  neutral review
  [0.0030]  positive review

Text     : Could you please reschedule our meeting to Thursday?
Template : The intent of this message is a {}.
  [0.8891]  request
  [0.0724]  complaint
  [0.0240]  greeting
  [0.0146]  farewell

Text     : Pursuant to the aforementioned clause, the party of the first part shall indemnify.
Template : This text belongs to the category of {}.
  [0.8760]  legal document
  [0.0590]  social media post
  [0.0354]  academic paper
  [0.0297]  casual conversation

Text     : Bhai kal match tha, kya scene tha yaar.
Template : This is an example of {}.
  [0.5759]  informal conversation
  [0.4083]  formal communication
  [0.0158]  technical documentation



In [5]:
# ── 4. cross-encoder/nli-deberta-v3-small ─────────────────────────────────────
#    Faster than BART, still accurate — good for latency-sensitive pipelines
#    Best for: production APIs, moderate accuracy requirements
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-small")

prompts = [
    ("Tanvi's blood pressure has been elevated for three weeks.",      ["medical concern", "financial issue", "travel update", "sports news"]),
    ("The pull request was merged after resolving merge conflicts.",   ["software development", "cooking", "politics", "music"]),
    ("Karan filed his ITR before the July deadline.",                  ["finance/tax", "sports", "technology", "entertainment"]),
]

print("── DeBERTa small (faster inference) ──")
for text, labels in prompts:
    result = pipe(text, candidate_labels=labels)
    print(f"Text   : {text}")
    for label, score in zip(result["labels"], result["scores"]): # type: ignore
        print(f"  [{score:.4f}]  {label}")
    print()

# Text   : Tanvi's blood pressure has been elevated for three weeks.
#   [0.9623]  medical concern
#   [0.0198]  financial issue
#   [0.0112]  travel update
#   [0.0067]  sports news

# Text   : The pull request was merged after resolving merge conflicts.
#   [0.9834]  software development
#   [0.0089]  politics
#   [0.0056]  cooking
#   [0.0021]  music


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Device set to use cpu


── DeBERTa small (faster inference) ──
Text   : Tanvi's blood pressure has been elevated for three weeks.
  [0.8850]  medical concern
  [0.0514]  travel update
  [0.0357]  sports news
  [0.0279]  financial issue

Text   : The pull request was merged after resolving merge conflicts.
  [0.4823]  software development
  [0.4537]  politics
  [0.0386]  cooking
  [0.0255]  music

Text   : Karan filed his ITR before the July deadline.
  [0.4112]  finance/tax
  [0.2883]  sports
  [0.2170]  technology
  [0.0835]  entertainment



In [9]:
import os
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN", "")
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
# ── 5. MoritzLaurer/deberta-v3-large-zeroshot-v2 ─────────────────────────────
#    Highest accuracy zero-shot model — heavier but best for critical decisions
#    Best for: legal, medical, compliance classification where accuracy matters
# ─────────────────────────────────────────────────────────────────────────────

# pipe = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-large-zeroshot-v2")
pipe = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0")


high_stakes_prompts = [
    (
        "The patient presents with acute chest pain radiating to the left arm.",
        ["cardiac emergency", "gastrointestinal issue", "anxiety attack", "musculoskeletal pain"]
    ),
    (
        "The accused was found in possession of forged documents.",
        ["criminal offense", "civil dispute", "regulatory violation", "contract breach"]
    ),
    (
        "Om's startup received a term sheet from a Series A investor.",
        ["fundraising", "product launch", "legal dispute", "hiring"]
    ),
]

print("── DeBERTa large (high accuracy, critical tasks) ──")
for text, labels in high_stakes_prompts:
    result = pipe(text, candidate_labels=labels)
    print(f"Text   : {text}")
    for label, score in zip(result["labels"], result["scores"]): # type: ignore
        print(f"  [{score:.4f}]  {label}")
    print()


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

Device set to use cpu


── DeBERTa large (high accuracy, critical tasks) ──
Text   : The patient presents with acute chest pain radiating to the left arm.
  [0.9926]  cardiac emergency
  [0.0046]  musculoskeletal pain
  [0.0022]  anxiety attack
  [0.0005]  gastrointestinal issue

Text   : The accused was found in possession of forged documents.
  [0.8895]  criminal offense
  [0.0672]  civil dispute
  [0.0398]  regulatory violation
  [0.0035]  contract breach

Text   : Om's startup received a term sheet from a Series A investor.
  [0.9937]  fundraising
  [0.0030]  legal dispute
  [0.0018]  product launch
  [0.0015]  hiring



In [12]:
# ── 6. batch processing ───────────────────────────────────────────────────────
#    Pass a list of strings — model processes them together, more efficient
#    Best for: classifying large datasets, log files, user feedback at scale
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

texts = [
    "Harsha joined a yoga class and has been meditating every morning.",
    "The RBI kept interest rates unchanged at 6.5 percent.",
    "The new transformer architecture reduced inference latency by 40 percent.",
    "Drishya won gold in the 100m sprint at the state championship.",
    "The court issued a stay order on the construction project.",
]

labels = ["health/wellness", "finance", "technology", "sports", "legal"]

print("── Batch classification ──")
results = pipe(texts, candidate_labels=labels)

for text, result in zip(texts, results): # type: ignore
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print(f"[{top_score:.4f}] {top_label:20s}  ←  {text}")
print()

# [0.9712] health/wellness       ←  Harsha joined a yoga class and has been meditating every morning.
# [0.9534] finance               ←  The RBI kept interest rates unchanged at 6.5 percent.
# [0.9623] technology            ←  The new transformer architecture reduced inference latency by 40 percent.
# [0.9812] sports                ←  Drishya won gold in the 100m sprint at the state championship.
# [0.9423] legal                 ←  The court issued a stay order on the construction project.

Device set to use cpu


── Batch classification ──
[0.9584] health/wellness       ←  Harsha joined a yoga class and has been meditating every morning.
[0.6812] finance               ←  The RBI kept interest rates unchanged at 6.5 percent.
[0.9510] technology            ←  The new transformer architecture reduced inference latency by 40 percent.
[0.8798] sports                ←  Drishya won gold in the 100m sprint at the state championship.
[0.4757] legal                 ←  The court issued a stay order on the construction project.

